# Analysis of Misclassifications

In [103]:
from pathlib import Path
import json
import pickle

import pandas as pd

In [104]:
# Experiment config

dataset = "darpa2000"
scenario = "s1_inside_dmz"

logic_file = "darpa_test"
train_mode = "scratch" # "scratch" or "pretrained"
dataset_variant = "down" # "original" or "resampled"
window_size = 10

run_id = "20260402_123738"

## Compute Misclassifications

### Load Original Flows

In [105]:
scenario_parts = scenario.split("_")
if len(scenario_parts) == 3:
    test_set_tag = f"{scenario_parts[0]}_{scenario_parts[2]}"
else:
    test_set_tag = scenario

In [106]:
df = pd.read_csv(
    f"../data/interim/{dataset}/{test_set_tag}/flows_labeled/all_flows_labeled.csv"
)

df = df.sort_values("start_time").reset_index(drop=True)
df['t_rel'] = df['start_time'] - df['start_time'].min()

### Load DPL Dataset

In [107]:
def load_dpl_dataset(logic_file, cache_file_name):
    dpl_dataset_dir = Path(f"../experiments/{dataset}/{scenario}/deepproblog/{logic_file}/cache/")
    cache_file_test = dpl_dataset_dir / cache_file_name

    cache = pickle.load(open(cache_file_test, "rb"))
    cache_df = pd.DataFrame(cache)
    cache_df.head()

    return cache_df

In [108]:
cache_file_name = f"{logic_file}_{dataset_variant}_w{window_size}_test.pkl"
cache_df = load_dpl_dataset(logic_file, cache_file_name)

### Map Misclassifications to Original Flows

In [109]:
errors_dir = f"../experiments/{dataset}/{scenario}/deepproblog/{logic_file}/errors"
experiment_name = f"{logic_file}_{train_mode}_{dataset_variant}_w{window_size}"

errors_file = (
    f"{errors_dir}/"
    f"{experiment_name}_{run_id}.json"
)
    
with open(errors_file) as f:
    errors = json.load(f)

In [110]:
dpl_to_orig = dict(zip(cache_df['dpl_index'], cache_df['orig_index']))

original_indices = []
mis_y_preds = []
mis_y_trues = []

phase_map = {
    "benign": 0,
    "phase1": 1,
    "phase2": 2,
    "phase3": 3,
    "phase4": 4,
    "phase5": 5,
}

for error in errors:
    dpl_index = error['index']

    original_indices.append(dpl_to_orig[dpl_index])
    y_pred = error['predicted']
    y_true = error['actual']
    mis_y_preds.append(phase_map[y_pred])
    mis_y_trues.append(phase_map[y_true])

mis_df = df.loc[original_indices].copy()
mis_df['y_pred'] = mis_y_preds
mis_df['y_true'] = mis_y_trues

In [111]:
mis_df

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
3164,f3200,9.524417e+08,9.524417e+08,0.121852,202.77.162.213,647,172.16.115.20,54791,udp,-,...,1,100,1,52,-,17,2,2831.281962,0,2
3261,f3282,9.524419e+08,9.524419e+08,0.471611,202.77.162.213,651,172.16.114.10,32779,udp,-,...,1,100,1,52,-,17,2,3011.790136,0,2
3264,f3284,9.524419e+08,9.524419e+08,0.568888,202.77.162.213,652,172.16.114.20,32773,udp,-,...,1,100,1,52,-,17,2,3012.274679,0,2
3266,f3286,9.524419e+08,9.524419e+08,0.550163,202.77.162.213,653,172.16.114.30,32772,udp,-,...,1,100,1,52,-,17,2,3012.858648,0,2
3558,f3627,9.524421e+08,9.524421e+08,0.058064,202.77.162.213,660,172.16.112.10,56255,udp,-,...,1,100,1,52,-,17,2,3253.948860,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42115,f42089,9.524473e+08,9.524473e+08,0.048319,172.16.115.5,49778,131.84.1.31,80,tcp,http,...,7,597,9,6530,-,6,5,8423.379288,0,5
42116,f42090,9.524473e+08,9.524473e+08,0.073494,172.16.115.5,49779,131.84.1.31,80,tcp,http,...,10,732,19,21877,-,6,5,8423.435351,0,5
42117,f42091,9.524473e+08,9.524473e+08,0.066602,172.16.115.5,49780,131.84.1.31,80,tcp,http,...,8,635,14,15078,-,6,5,8423.533364,0,5
42118,f42092,9.524473e+08,9.524473e+08,0.036762,172.16.115.5,49781,131.84.1.31,80,tcp,http,...,6,558,7,5122,-,6,5,8423.613914,0,5


## Misclassification Analysis

### Helper functions

In [112]:
def false_positives(df, phase):
    return df[(df["y_true"] != phase) & (df["y_pred"] == phase)]

def false_negatives(df, phase):
    return df[(df["y_true"] == phase) & (df["y_pred"] != phase)]

def true_positives(df, phase):
    return df[(df["y_true"] == phase) & (df["y_pred"] == phase)]

### Per-Phase Misclassifications

#### Phase 2

In [113]:
df_fp_2 = false_positives(mis_df, 2)
df_fp_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


In [114]:
df_fn_2 = false_negatives(mis_df, 2)
df_fn_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
3164,f3200,9.524417e+08,9.524417e+08,0.121852,202.77.162.213,647,172.16.115.20,54791,udp,-,...,1,100,1,52,-,17,2,2831.281962,0,2
3261,f3282,9.524419e+08,9.524419e+08,0.471611,202.77.162.213,651,172.16.114.10,32779,udp,-,...,1,100,1,52,-,17,2,3011.790136,0,2
3264,f3284,9.524419e+08,9.524419e+08,0.568888,202.77.162.213,652,172.16.114.20,32773,udp,-,...,1,100,1,52,-,17,2,3012.274679,0,2
3266,f3286,9.524419e+08,9.524419e+08,0.550163,202.77.162.213,653,172.16.114.30,32772,udp,-,...,1,100,1,52,-,17,2,3012.858648,0,2
3558,f3627,9.524421e+08,9.524421e+08,0.058064,202.77.162.213,660,172.16.112.10,56255,udp,-,...,1,100,1,52,-,17,2,3253.948860,0,2
3560,f3629,9.524421e+08,9.524421e+08,0.085522,202.77.162.213,661,172.16.112.50,56261,udp,-,...,1,100,1,52,-,17,2,3254.024577,0,2


In [115]:
df_fn_2["local_orig"]
df_fn_2["local_resp"]

3164    T
3261    T
3264    T
3266    T
3558    T
3560    T
Name: local_resp, dtype: object

In [116]:
df_tp_2 = true_positives(mis_df, 2)
df_tp_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


#### Phase 3

In [117]:
df_fn_3 = false_negatives(mis_df, 3)
df_fn_3

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
4572,f4556,9.524432e+08,9.524432e+08,8.269662,202.77.162.213,1425,172.16.115.20,23,tcp,-,...,13,566,13,639,-,6,3,4338.577052,0,3
4582,f4557,9.524432e+08,9.524432e+08,8.309972,202.77.162.213,1488,172.16.115.20,23,tcp,-,...,13,566,13,639,-,6,3,4346.846806,0,3
4587,f4559,9.524432e+08,9.524432e+08,1.595575,202.77.162.213,1495,172.16.115.20,23,tcp,-,...,13,566,13,679,-,6,3,4355.157041,0,3
4593,f4563,9.524432e+08,9.524432e+08,4.256986,202.77.162.213,1496,172.16.114.10,23,tcp,-,...,14,606,14,679,-,6,3,4356.753117,0,3
4606,f4569,9.524432e+08,9.524432e+08,4.222506,202.77.162.213,1497,172.16.114.10,23,tcp,-,...,13,566,13,639,-,6,3,4360.974895,0,3
4611,f4574,9.524432e+08,9.524432e+08,8.201315,202.77.162.213,1498,172.16.114.10,23,tcp,-,...,13,566,13,639,-,6,3,4365.161746,0,3
4620,f4577,9.524432e+08,9.524432e+08,8.803595,202.77.162.213,1503,172.16.114.20,23,tcp,-,...,14,606,14,679,-,6,3,4373.357103,0,3
4628,f4582,9.524432e+08,9.524432e+08,8.446004,202.77.162.213,1504,172.16.114.20,23,tcp,-,...,14,606,14,679,-,6,3,4382.097176,0,3
4634,f4588,9.524432e+08,9.524433e+08,8.417312,202.77.162.213,1505,172.16.114.20,23,tcp,-,...,14,606,14,679,-,6,3,4390.477365,0,3
4641,f4616,9.524433e+08,9.524433e+08,8.868900,202.77.162.213,1506,172.16.114.30,23,tcp,-,...,14,606,14,679,-,6,3,4398.867885,0,3


In [118]:
df_fp_3 = false_positives(mis_df, 3)
df_fp_3

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true


#### Phase 4

In [119]:
df_fn_4 = false_negatives(mis_df, 4)
df_fn_4

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
5516,f5910,9.524442e+08,9.524442e+08,0.001277,172.16.112.10,1023,202.77.162.213,1018,tcp,-,...,2,100,1,44,-,6,4,5381.815260,0,4
5522,f5949,9.524443e+08,9.524443e+08,0.001377,172.16.112.50,1023,202.77.162.213,1016,tcp,-,...,2,100,1,44,-,6,4,5397.844729,0,4


In [120]:
df_fp_4 = false_positives(mis_df, 4)
df_fp_4

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
4726,f5471,9.524433e+08,9.524442e+08,900.429692,202.77.162.213,1535,172.16.112.50,23,tcp,-,...,15,646,14,1873,-,6,0,4445.238014,4,0
5180,f45437,9.524438e+08,9.524506e+08,6808.971401,197.182.91.233,13449,172.16.115.20,23,tcp,-,...,3171,128508,1646,75839,-,6,0,4936.829975,4,0


In [121]:
df_fp_4["dport"].value_counts()

dport
23    2
Name: count, dtype: int64

#### Phase 5

In [122]:
df_fn_5 = false_negatives(mis_df, 5)
df_fn_5

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,phase,t_rel,y_pred,y_true
7507,f45282,9.524464e+08,9.524469e+08,485.693936,202.77.162.213,1566,172.16.115.20,23,tcp,-,...,305,12519,176,8664,-,6,5,7519.353413,0,5
7562,f7538,9.524464e+08,9.524464e+08,0.001545,172.16.115.20,1022,202.77.162.213,1020,tcp,-,...,2,80,2,80,-,6,5,7568.456301,0,5
7592,f7569,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,4047,8.138.161.2,4939,tcp,-,...,1,40,0,0,-,6,5,7615.068694,0,5
7593,f7570,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,31938,61.56.80.155,4940,tcp,-,...,1,40,0,0,-,6,5,7615.068855,0,5
7594,f7571,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,6439,116.107.159.190,4941,tcp,-,...,1,40,0,0,-,6,5,7615.069198,0,5
7595,f7572,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,24037,45.4.65.127,4942,tcp,-,...,1,40,0,0,-,6,5,7615.069964,0,5
7596,f7573,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,19241,119.93.67.81,4943,tcp,-,...,1,40,0,0,-,6,5,7615.070060,0,5
7597,f7574,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,3537,83.196.141.150,4944,tcp,-,...,1,40,0,0,-,6,5,7615.070195,0,5
7598,f7575,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,14824,2.178.110.11,4945,tcp,-,...,1,40,0,0,-,6,5,7615.070290,0,5
7599,f7576,9.524465e+08,9.524465e+08,0.000000,131.84.1.31,15303,65.143.237.207,4946,tcp,-,...,1,40,0,0,-,6,5,7615.070402,0,5
